# OSIS Quickstart Tutorial

This interactive tutorial demonstrates the core capabilities of the **Optical Storage Intelligence Simulator (OSIS)**:
1. Simulating standardized optical disc formats (CD-RW, DVD-RW, Blu-ray BD-RE)
2. Multilayer thin-film stack reflectance via the Abelès Transfer Matrix Method (TMM)
3. Vectorized parameter sweeps across physical variables
4. One-At-a-Time (OAT) sensitivity analysis and parameter elasticity rankings

In [ ]:
import osis
import numpy as np
import matplotlib.pyplot as plt

## 1. Simulate Standard Optical Disc Readout Channel

In [ ]:
# Run simulation for Blu-ray BD-RE
bd_cfg = osis.BluRayConfig()
res = osis.simulate(bd_cfg)

print(f"Format: {bd_cfg.name}")
print(f"Laser Wavelength: {bd_cfg.wavelength_m * 1e9:.1f} nm")
print(f"Numerical Aperture: {bd_cfg.numerical_aperture:.2f}")
print(f"Airy Spot Radius: {res['spot_radius_m'] * 1e9:.1f} nm")
print(f"Land Reflectance: {res['r_land'] * 100:.2f} %")
print(f"Mark Reflectance: {res['r_mark'] * 100:.2f} %")
print(f"MTF factor at T_min: {res['mtf']:.3f}")
print(f"Carrier-to-Noise Ratio (CNR): {res['cnr_db']:.2f} dB")
print(f"Estimated Bit Error Rate (BER): {res['ber']:.2e}")

## 2. Compare CD, DVD, and Blu-ray Readout Performance

In [ ]:
presets = [
    ("CD-RW", osis.CDConfig()),
    ("DVD-RW", osis.DVDConfig()),
    ("Blu-ray BD-RE", osis.BluRayConfig()),
]

print(f"{'Standard':<16} | {'λ (nm)':<8} | {'NA':<6} | {'Spot (nm)':<10} | {'MTF':<6} | {'CNR (dB)':<8}")
print("-" * 65)
for name, cfg in presets:
    r = osis.simulate(cfg)
    print(f"{name:<16} | {cfg.wavelength_m * 1e9:>6.1f}   | {cfg.numerical_aperture:>4.2f}   | {r['spot_radius_m'] * 1e9:>8.1f}   | {r['mtf']:>6.3f} | {r['cnr_db']:>7.2f}")

## 3. Parameter Sweep: Numerical Aperture effect on CNR

In [ ]:
na_vals = np.linspace(0.70, 0.90, 11)
sweep = osis.parameter_sweep(bd_cfg, "numerical_aperture", na_vals)

plt.figure(figsize=(7, 4), dpi=120)
plt.plot([s['value'] for s in sweep], [s['cnr_db'] for s in sweep], 'o-', color='#2ca02c', lw=2)
plt.title('Blu-ray BD-RE: Carrier-to-Noise Ratio vs. Numerical Aperture')
plt.xlabel('Numerical Aperture (NA)')
plt.ylabel('CNR (dB)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Sensitivity Analysis & Elasticity Ranking

In [ ]:
sens = osis.one_at_a_time_sensitivity(bd_cfg, delta_fraction=0.05)
ranked = osis.rank_parameters_by_influence(sens, metric='elasticity')

print(f"{'Rank':<4} | {'Parameter':<24} | {'Elasticity':<12} | {'CNR Swing (dB)':<14}")
print("-" * 60)
for rank, (param, _) in enumerate(ranked[:6], start=1):
    m = sens[param]
    print(f"{rank:<4} | {param:<24} | {m['elasticity']:>+10.4f}  | {m['delta_output']:>11.3f} dB")